# Parametric continuous spectrum via an auxiliary network

This notebook demonstrates `koopman_parameterization="auxiliary_spectral"` on
`ContinuousKoopmanOperator`: an MLP maps latent state $z$ to instantaneous
eigenvalues and assembles a **block-diagonal rotation–scaling generator**
$L(z)$ (Lusch, Kutz & Brunton, *Nature Communications* 2018).

## Honest framing (read first)

- This is a **parametric / locally linear** spectrum — **not** a fixed global
  Koopman matrix. State-dependent eigenvalues weaken global spectral-radius
  and Hurwitz certificates.
- Prefer **delay embeddings** (`examples/17_delay_embedding_partial_observability.ipynb`)
  as the first continuous-spectrum tool in practice; this mode is complementary.
- Do **not** overclaim recovery of the continuous spectrum of the
  *infinite-dimensional* Koopman operator.

## What we show

1. Synthetic amplitude-dependent oscillator (frequency rises with radius).
2. Auxiliary mode vs fixed dense generator at matched latent dim.
3. Instantaneous frequency $\omega(z)$ correlating with amplitude.


In [ ]:
import torch
from torch import nn

from koopman_graph import ContinuousKoopmanOperator

torch.manual_seed(0)
device = torch.device("cpu")

## 1. Amplitude-dependent oscillator data

Local generator frozen at the start of each step:

$$
L(r) =
\begin{bmatrix}
-\gamma & -\omega(r) \\
\omega(r) & -\gamma
\end{bmatrix},
\qquad
\omega(r) = 1 + 0.75\, r,
\quad r=\|z\|.
$$

In [ ]:
def make_pairs(n_pairs=512, delta_t=0.1, seed=0):
    g = torch.Generator().manual_seed(seed)
    radii = 0.5 + 1.5 * torch.rand(n_pairs, generator=g)
    angles = 2 * torch.pi * torch.rand(n_pairs, generator=g)
    z0 = torch.stack([radii * torch.cos(angles), radii * torch.sin(angles)], dim=-1)
    omega = 1.0 + 0.75 * radii
    gamma = 0.05
    z1 = []
    for i in range(n_pairs):
        w = float(omega[i])
        L = torch.tensor([[-gamma, -w], [w, -gamma]])
        z1.append(z0[i] @ torch.linalg.matrix_exp(L * delta_t).T)
    return z0, torch.stack(z1), radii, omega


delta_t = 0.1
z0, z1, radii, true_omega = make_pairs()
z0, z1 = z0.to(device), z1.to(device)
print(z0.shape, "delta_t=", delta_t)

## 2. Fit auxiliary vs dense generators

Hidden widths are configurable via `auxiliary_hidden_dims` (default `(64, 64)`).
On `GraphKoopmanModel`, pass `koopman_auxiliary_hidden_dims=...`.

In [ ]:
def fit_op(op, z0, z1, *, epochs=200, lr=1e-2):
    opt = torch.optim.Adam(op.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.functional.mse_loss(op.advance(z0, delta_t), z1)
        loss.backward()
        opt.step()
    return float(loss.detach())


def long_horizon_mse(op, starts, horizon=20):
    roll = starts.clone()
    target = starts.clone()
    with torch.no_grad():
        for _ in range(horizon):
            r = target.norm(dim=-1)
            w = 1.0 + 0.75 * r
            nxt = []
            for i in range(target.shape[0]):
                L = torch.tensor([[-0.05, -float(w[i])], [float(w[i]), -0.05]])
                nxt.append(target[i] @ torch.linalg.matrix_exp(L * delta_t).T)
            target = torch.stack(nxt)
            roll = op.advance(roll, delta_t)
    return float(nn.functional.mse_loss(roll, target))


torch.manual_seed(0)
aux = ContinuousKoopmanOperator(
    2,
    parameterization="auxiliary_spectral",
    auxiliary_hidden_dims=(64, 64),
    init_mode="identity",
).to(device)
torch.manual_seed(0)
dense = ContinuousKoopmanOperator(2, parameterization="dense", init_mode="identity").to(
    device
)

print("aux one-step MSE", fit_op(aux, z0, z1))
print("dense one-step MSE", fit_op(dense, z0, z1))
starts = z0[:32]
aux_h = long_horizon_mse(aux, starts)
dense_h = long_horizon_mse(dense, starts)
print(f"long-horizon MSE  aux={aux_h:.4f}  dense={dense_h:.4f}")
assert aux_h < dense_h, "auxiliary mode should beat fixed dense on this system"

## 3. Instantaneous frequency vs amplitude

Use `generator_at(z)` / the auxiliary net — **not** a global `matrix` / `L`
(those raise in this mode).

In [ ]:
with torch.no_grad():
    _mu, omega_hat, _ = aux.auxiliary_net(z0)
    omega_hat = omega_hat.reshape(-1)
    pearson = torch.corrcoef(torch.stack([radii.to(device), omega_hat]))[0, 1]
    L_sample = aux.generator_at(z0[0])
    eig = aux.instantaneous_spectrum(z0[0])

print(f"Pearson(r, ω̂) = {float(pearson):.3f}  (true ω also rises with r)")
print("sample L(z):\n", L_sample)
print("instantaneous eigenvalues:", eig)
assert float(pearson) > 0.3

# Controlled advance is supported (locally linear freeze of L(z) at step start).
ctrl = ContinuousKoopmanOperator(
    2,
    parameterization="auxiliary_spectral",
    control_dim=1,
    control_mode="additive",
    auxiliary_hidden_dims=(32,),
)
z_c = torch.tensor([1.0, 0.0])
u = torch.tensor([0.1])
z_next = ctrl.advance(z_c, 0.05, control=u)
print("controlled step", z_next)

## Takeaways

- `auxiliary_spectral` gives a **state-dependent** continuous generator with
  `generator_at(z)` / `instantaneous_spectrum(z)`.
- Cite Lusch et al. (2018) for the auxiliary-network idea; keep claims local.
- For continuous-spectrum *phenomenology* with partial observations, start with
  delay embeddings (notebook 17), then consider this parameterization when a
  locally linear spectrum is the right inductive bias.
